<h1>🎪<b><i> Live Class Activities — Teacher Console</i></b></h1>

### Wisdom of the Crowd 🔮 &nbsp;+&nbsp; The Strategy Arena 🏆

This notebook turns **your** Colab runtime into a little web server that students join
from their phones or laptops. Each activity is one cell: run it, and you get two links —

- 📣 a **student link** to share with the class (works on any device, no accounts), and
- 🔑 a **teacher link** you open yourself and put on the projector.

> 🔒 **Keep this notebook private** — your teacher links contain the key that controls
> the activities.
>
> 🖥️ **Runtime:** any runtime works (an A100 is plenty — the servers are tiny). The
> links live only as long as this runtime does.


---
## Table of Contents

> **0. Setup** — fetch the activity server (one cell, ~20 s)
>
> **1. Wisdom of the Crowd** 🔮 — students guess numbers; you reveal the truth, the
> crowd average, and the full distribution, one question at a time
>
> **2. The Strategy Arena** 🏆 — students *describe* a strategy in plain English, an
> LLM turns it into code, they test-drive it privately, then enter the competition;
> you run the market and watch every strategy race across the screen
>
> **3. The Lecture Dashboard** 📊 — animated, play-button versions of the lab's key
> charts for use during the lecture
>
> **4. Troubleshooting & notes**


---
<!-- ⏱ Suggested time: 1 min -->
## 0. Setup

Run this once per session. It pulls the activity server code from the class GitHub repo
and downloads the tunnel client that makes your Colab reachable from student devices.

*(The repo must be public for the clone to work — or upload `classroom.py` by hand via
the Files pane on the left.)*


In [ ]:
#@title 🏗 Setup Cell — run once per session { display-mode: "form" }
REPO = "https://github.com/bwathomas/ml-trading-intro"  #@param {type:"string"}

import os, sys
if not os.path.exists("/content/ml-trading-intro"):
    !git clone -q $REPO /content/ml-trading-intro
else:
    !cd /content/ml-trading-intro && git pull -q

sys.path.insert(0, "/content/ml-trading-intro")
from classroom import launch_wisdom, launch_arena, launch_dashboard

print("✅ Ready. Run an activity cell below when class starts.")


---
<!-- ⏱ Suggested time: 10–15 min in class -->
## 1. Wisdom of the Crowd 🔮

The lecture's opening demo, live: in 1906 a crowd of county-fair visitors guessed the
weight of an ox almost perfectly — now your class gets to be the crowd.

**Run of show:**

1. *(Optional)* edit the question list in the next cell — each entry is
   `("question", true_answer, "unit")`.
2. Run the **launch cell**. Share the *student link* (paste it in the class chat, or
   put it on a slide). Open the *teacher link* yourself and project it.
3. Students see one question at a time and type in a number. Your screen shows a live
   submission counter.
4. Click **🎉 Reveal the answer** — the projector shows the true answer, the crowd's
   average and median, and a histogram of every guess (the green bar holds the truth).
   Students see the same reveal on their own devices.
5. Click **Next question →** and repeat. The classic result: the crowd's *average*
   beats almost every *individual* — discuss why that's bad news for stock pickers.


In [ ]:
# (Optional) edit the questions, then run this cell BEFORE launching.
# Skip it entirely to use these six defaults.
QUESTIONS = [
    ("In 1906, 787 visitors to an English county fair each guessed the weight "
     "of an ox. Francis Galton expected nonsense. Your turn: how much does a "
     "prize ox weigh, in pounds?", 1198, "lbs"),
    ("How many jellybeans fit in a classic one-gallon jar?", 930, "jellybeans"),
    ("How long is the Golden Gate Bridge (total, including approaches), in feet?", 8980, "feet"),
    ("How far away is the Moon, on average, in miles?", 238855, "miles"),
    ("What is the population of Australia, in millions?", 27.2, "million"),
    ("How tall is Mount Everest, in feet?", 29032, "feet"),
]
print(f"{len(QUESTIONS)} questions loaded")


In [ ]:
#@title 🔮 Launch Activity 1 — Wisdom of the Crowd { display-mode: "form" }
TEACHER_KEY = "change-me"  #@param {type:"string"}

launch_wisdom(questions=globals().get("QUESTIONS"), teacher_key=TEACHER_KEY);


---
<!-- ⏱ Suggested time: 15–20 min in class -->
## 2. The Strategy Arena 🏆

The grand finale: every student designs a strategy and the strategies battle on a
market replay nobody has been graded on. You control the battlefield with the knobs
below — which stocks, which dates, what fees.

**How students play:** on the student page they type a team name and **describe their
strategy in plain English**. The server sends the description to an LLM (your OpenAI
key), turns it into a `Strategy` subclass, and immediately **test-drives it on a
practice window** — the student sees a private graph of how their idea performs, plus
ROI and Sharpe. They can iterate on the prompt as much as they like; when happy, they
hit **🚀 Submit** to enter (or update) their entry in the competition. There's also a
"paste code directly" fallback for students who built one by hand in Part 6 of the lab.

**Run of show:**

1. Store your OpenAI key once as a Colab **secret named `OPENAI_API_KEY`** (🔑 panel on
   the left, toggle on notebook access), set the knobs, and run the launch cell. The
   market data downloads first (~1 min for the full S&P 500), then the links appear.
2. Share the *student link*. Your console shows who has entered.
3. When entries stop, click **🔒 Lock submissions**, then **🎬 RUN THE MARKET**. Every
   entry is replayed day by day on the *competition year* — a different random year
   than the one they practiced on (and only ever seeing the past — no oracles
   allowed).
4. The race animation plays on the projector: every strategy's profit line sweeps
   across the replay window together. When the lines hit the final day, the podium
   appears: **top 3 names with their ROI**. 🥇🥈🥉
5. Everyone else checks *their own device* — each student privately sees their rank,
   ROI, and Sharpe (or their crash message). Nobody else can see it.

**The windows:** by default the arena uses the **full S&P 500** and picks two random
calendar years (≥ `YEAR_MIN`, default 2016): students test-drive on one year, and the
competition replays a **different** year — so nobody can overfit the scoring window,
and every class gets a fresh market (will it be a 2017 grind-up or a 2022 bloodbath?).
The years are revealed on the podium. Turn `RANDOM_YEARS` off to pin manual dates
instead. Strategies get `WARMUP_DAYS` of history before each window; `FEE` is charged
on every dollar of position changed per day, so hyperactive strategies bleed. An empty
`OPENAI_API_KEY` disables prompt-mode (paste-only).

> ⚠️ The arena executes student-submitted Python on this runtime (in a subprocess,
> with a hard timeout). That's fine on a throwaway Colab VM — just don't run it
> somewhere you keep things you love, and don't share the teacher key.


In [ ]:
#@title 📈 Launch Activity 2 — The Strategy Arena { display-mode: "form" }
#@markdown The OpenAI key is read from the Colab secret `OPENAI_API_KEY` (🔑 panel on the
#@markdown left — add it once, grant this notebook access). The field below is only a
#@markdown fallback if you prefer pasting it.
TEACHER_KEY = "change-me"          #@param {type:"string"}
OPENAI_API_KEY = ""                #@param {type:"string"}
LLM_MODEL = "gpt-5-mini"           #@param ["gpt-5-mini", "gpt-5", "gpt-4o-mini"]

if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        print("⚠️ no OPENAI_API_KEY secret found — prompt-mode disabled, paste-code only")
UNIVERSE = "S&P 500"               #@param ["S&P 500", "Dow 30", "Custom (list below)"]
CUSTOM_TICKERS = "AAPL, MSFT, NVDA, AMZN, GOOGL, META, TSLA, JPM, XOM, KO"  #@param {type:"string"}
#@markdown Random-year mode (default): practice runs on one random calendar year ≥ `YEAR_MIN`, the competition on a *different* random year.
RANDOM_YEARS = True                #@param {type:"boolean"}
YEAR_MIN = 2016                    #@param {type:"integer"}
#@markdown Manual dates (used only if `RANDOM_YEARS` is off):
REPLAY_START = "2026-01-02"        #@param {type:"date"}
REPLAY_END = "2026-06-10"          #@param {type:"date"}
PRACTICE_DAYS = 60                 #@param {type:"integer"}
WARMUP_DAYS = 20                   #@param {type:"integer"}
FEE = 0.0005                       #@param {type:"number"}
STRATEGY_TIMEOUT_S = 60            #@param {type:"integer"}

launch_arena(universe=UNIVERSE, custom_tickers=CUSTOM_TICKERS,
             random_years=RANDOM_YEARS, year_min=YEAR_MIN,
             replay_start=REPLAY_START, replay_end=REPLAY_END,
             warmup=WARMUP_DAYS, fee=FEE, teacher_key=TEACHER_KEY,
             strategy_timeout=STRATEGY_TIMEOUT_S,
             openai_api_key=OPENAI_API_KEY, llm_model=LLM_MODEL,
             practice_days=PRACTICE_DAYS);


---
<!-- ⏱ Suggested time: used throughout the lecture -->
## 3. The Lecture Dashboard 📊

Animated versions of the lab's key charts, for the lecture itself. Each panel has a
**▶ Play** button that sweeps the lines across the screen left-to-right (and **⏭ End**
to jump to the finished chart):

1. **Growth of \$1, buy & hold** (log scale)
2. **\$1 every day**, in at the open, out at the close
3. **Online learning mini-arena**: uniform vs. Follow-The-Leader vs. the oracle
4. **ARIMA forecasting past a cutoff**, with its 95% confidence cone vs. reality
   (per-ticker selector)

The box at the top of the page controls **which tickers** are shown and **the forecast
cutoff date**; *Apply & rebuild* re-downloads and recomputes everything (~30 s). The
controls only work from your keyed link — the plain link is view-only, so you can also
share it with students if you like.


In [ ]:
#@title 📊 Launch the Lecture Dashboard { display-mode: "form" }
TEACHER_KEY = "change-me"          #@param {type:"string"}
TICKERS = "^GSPC, BTC-USD, GME"    #@param {type:"string"}
START = "2019-01-01"               #@param {type:"date"}
FORECAST_CUTOFF = "2025-06-01"     #@param {type:"date"}

launch_dashboard(tickers=TICKERS, start=START, cutoff=FORECAST_CUTOFF,
                 teacher_key=TEACHER_KEY);


---
## 4. Troubleshooting & notes

- **Links stop working?** They live only as long as this Colab runtime. If the runtime
  restarts, run setup + the activity cell again (you'll get fresh links).
- **Re-running an activity cell** restarts that activity from scratch (submissions are
  wiped) and prints new links — handy between class periods.
- **The tunnel** is a free Cloudflare quick-tunnel (`*.trycloudflare.com`) — no account
  needed, nothing to configure. If it ever fails to come up, just re-run the cell.
- **Don't test the links from a Colab cell** — the Colab VM often can't resolve
  fresh `trycloudflare.com` hostnames from inside itself. Open them from your own
  browser or phone; they work fine from outside.
- **Students on school Wi-Fi** occasionally have `trycloudflare.com` blocked; phone
  hotspots work in a pinch.
- **Wrong question/answer mid-activity?** Use **← Back**, or edit `QUESTIONS` and
  re-run both Activity-1 cells.
- **OpenAI key & cost:** the Arena's prompt mode makes one chat-completions call per
  "Generate" click (rate-limited to one per student per 8 s). A class session costs
  pennies on a mini-tier model. The key stays on the server — students never see it.
- **Arena fairness:** strategies only ever receive returns from days *before* the
  simulated "today," so pasting a future-peeking strategy is structurally impossible.
  Crashes and infinite loops just turn into a 💥 DNF for that team.
